# Forecasting Feature Evaluation

## Business Objective

Evaluate whether calendar and promotional features
improve monthly retail sales forecasting accuracy.

## Candidate Features

- Calendar month
- Number of calendar days
- Promotional activity, where known at forecast time

## Evaluation Principles

1. Use chronological training and validation windows.
2. Compare against existing baseline forecasting methods.
3. Prevent future-data leakage.
4. Evaluate WAPE, MAE, and RMSE.
5. Recommend features only when out-of-sample results
   support their usefulness.

## Important Limitation

Actual future promotional activity cannot be used
as a forecasting input unless it would genuinely
be known at the forecast creation date.

## 1. Existing Forecasting Framework Review

Before developing feature-based forecasting models,
inspect the existing forecasting notebook.

Identify:
- Forecast horizon
- Number of rolling-origin windows
- Training and test structure
- Existing forecasting models
- Evaluation metrics
- Saved forecasting reports

The new models must use the same evaluation windows
and target definition for a fair comparison.

In [38]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data" / "processed").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Project root not found.")
    PROJECT_ROOT = PROJECT_ROOT.parent

existing_notebook = (
    PROJECT_ROOT
    / "notebooks"
    / "05_forecasting"
    / "01_forecasting_and_evaluation.ipynb"
)

assert existing_notebook.exists(), "Existing forecasting notebook not found."

with open(existing_notebook, encoding="utf-8") as f:
    notebook_data = json.load(f)

print("Existing forecasting notebook found.")
print("Total cells:", len(notebook_data["cells"]))

for index, cell in enumerate(notebook_data["cells"]):
    if cell["cell_type"] == "markdown":
        content = "".join(cell["source"]).strip()
        heading = next(
            (line for line in content.splitlines() if line.startswith("#")),
            None
        )

        if heading:
            print(f"Cell {index}: {heading[:120]}")

Existing forecasting notebook found.
Total cells: 140
Cell 0: # Retail Sales Forecasting and Evaluation
Cell 2: ## Phase 3.9 — Forecasting Readiness & Time-Series Validation Design
Cell 3: ### Step 3.9.2 — Inspect the available forecasting history
Cell 6: ### Step 3.9.3 — Create the training and validation datasets
Cell 17: ### Phase 3.10 — Baseline Forecasting
Cell 22: ### Step 3.10.3 — Create the Seasonal Naive Forecast
Cell 24: ### Step 3.10.4 — Baseline Forecast QA
Cell 27: ## Step 3.10.5 — Diagnose and correct the QA
Cell 29: ### 2. Compare the actual dates
Cell 31: ### 3. Standardize both forecast indexes
Cell 33: ### 4. Rerun the QA
Cell 35: ### Step 3.10.6 — Calculate Forecast Evaluation Metrics
Cell 36: ### Step 3.10.6A — Prepare a common evaluation dataset
Cell 38: ### Step 3.10.6B — Calculate individual forecast errors
Cell 41: ### Baseline Forecast Evaluation — Initial Holdout
Cell 42: ### Step 3.11.1 — First, define the rolling-validation design
Cell 43: ## Phase 3.11 — Ro

## 2. Rolling-Origin Evaluation Design

Inspect the existing rolling-origin validation code
before implementing feature-based forecasting models.

The objective is to reuse the same:
- Monthly target series
- Forecast origins
- Forecast horizon
- Evaluation dates
- Error metrics

This ensures a fair comparison with existing models.

In [39]:
# Inspect the existing rolling-validation implementation

for index in range(44, 73):
    cell = notebook_data["cells"][index]

    if cell["cell_type"] == "code":
        source = "".join(cell["source"])

        print("\n" + "=" * 65)
        print(f"CELL {index} — CODE")
        print("=" * 65)
        print(source[:2500])


CELL 46 — CODE
monthly_ts["total_sales"]

CELL 47 — CODE
print("Number of observations:", len(monthly_ts))
print("Start:", monthly_ts["month"].min())
print("End:", monthly_ts["month"].max())

CELL 48 — CODE
monthly_ts[["month", "total_sales"]].head()

CELL 49 — CODE
monthly_ts[["month", "total_sales"]].tail()

CELL 51 — CODE
forecast_horizon = 3
minimum_training_months = 36

print("Forecast horizon:", forecast_horizon, "months")
print("Minimum training history:", minimum_training_months, "months")

CELL 53 — CODE
rolling_windows = []

n_observations = len(monthly_ts)

for train_end in range(
    minimum_training_months,
    n_observations - forecast_horizon + 1
):
    validation_start = train_end
    validation_end = train_end + forecast_horizon

    train_data = monthly_ts.iloc[:train_end].copy()
    validation_data = monthly_ts.iloc[
        validation_start:validation_end
    ].copy()

    rolling_windows.append({
        "train_start": train_data["month"].min(),
        "train_end

## 3. Reconstruct the Existing Rolling-Origin Windows

### Objective

Reproduce the original forecasting evaluation design.

### Parameters

- Minimum training history: 36 months
- Forecast horizon: 3 months
- Window advancement: 1 month
- Expected rolling windows: 17
- Expected forecast cases: 51

### Validation Principle

All training observations must occur before
their corresponding validation observations.

In [40]:
import pandas as pd

monthly_path = (
    PROJECT_ROOT
    / "reports"
    / "retail_driver_analysis"
    / "monthly_sales_promotion_performance.csv"
)

monthly_data = pd.read_csv(
    monthly_path,
    parse_dates=["month"]
)

monthly_data = (
    monthly_data.sort_values("month")
    .reset_index(drop=True)
)

monthly_data = monthly_data.rename(
    columns={"monthly_sales": "total_sales"}
)

forecast_horizon = 3
minimum_training_months = 36

rolling_windows = []

for train_end in range(
    minimum_training_months,
    len(monthly_data) - forecast_horizon + 1
):
    training = monthly_data.iloc[:train_end]
    validation = monthly_data.iloc[
        train_end:train_end + forecast_horizon
    ]

    rolling_windows.append({
        "window_id": len(rolling_windows) + 1,
        "train_start": training["month"].min(),
        "train_end": training["month"].max(),
        "validation_start": validation["month"].min(),
        "validation_end": validation["month"].max(),
        "train_rows": len(training),
        "validation_rows": len(validation)
    })

rolling_windows_df = pd.DataFrame(rolling_windows)

display(rolling_windows_df.head())
display(rolling_windows_df.tail())

,window_id,train_start,train_end,validation_start,validation_end,train_rows,validation_rows
0,1,2013-01-01,2015-12-01,2016-01-01,2016-03-01,36,3
1,2,2013-01-01,2016-01-01,2016-02-01,2016-04-01,37,3
2,3,2013-01-01,2016-02-01,2016-03-01,2016-05-01,38,3
3,4,2013-01-01,2016-03-01,2016-04-01,2016-06-01,39,3
4,5,2013-01-01,2016-04-01,2016-05-01,2016-07-01,40,3


,window_id,train_start,train_end,validation_start,validation_end,train_rows,validation_rows
12,13,2013-01-01,2016-12-01,2017-01-01,2017-03-01,48,3
13,14,2013-01-01,2017-01-01,2017-02-01,2017-04-01,49,3
14,15,2013-01-01,2017-02-01,2017-03-01,2017-05-01,50,3
15,16,2013-01-01,2017-03-01,2017-04-01,2017-06-01,51,3
16,17,2013-01-01,2017-04-01,2017-05-01,2017-07-01,52,3


In [41]:
assert len(monthly_data) == 55
assert monthly_data["month"].is_unique

assert len(rolling_windows_df) == 17

assert (
    rolling_windows_df["train_rows"] >= 36
).all()

assert (
    rolling_windows_df["validation_rows"] == 3
).all()

assert (
    rolling_windows_df["train_end"]
    < rolling_windows_df["validation_start"]
).all()

assert (
    rolling_windows_df["validation_start"]
    .is_monotonic_increasing
)

assert (
    rolling_windows_df["validation_rows"].sum()
    == 51
)

print("Rolling-origin reconstruction validation passed.")
print("Windows:", len(rolling_windows_df))
print(
    "Forecast cases:",
    rolling_windows_df["validation_rows"].sum()
)
print(
    "First validation:",
    rolling_windows_df["validation_start"].min()
)
print(
    "Last validation:",
    rolling_windows_df["validation_end"].max()
)

Rolling-origin reconstruction validation passed.
Windows: 17
Forecast cases: 51
First validation: 2016-01-01 00:00:00
Last validation: 2017-07-01 00:00:00


## 4. Forecasting Target Reconciliation

### Business Objective

Confirm that the monthly sales data used for
feature-based forecasting matches the original
forecasting target.

### Validation Rules

1. Both datasets contain 55 months.
2. Both datasets cover January 2013 to July 2017.
3. Monthly dates are unique.
4. Monthly sales values match within a small
   floating-point tolerance.
5. No monthly sales values are missing.

### Why This Matters

A fair forecasting comparison requires the same
target values, evaluation dates, and metrics.

In [42]:
import numpy as np
import pandas as pd

sales_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "train_standardized.csv"
)

sales_data = pd.read_csv(
    sales_path,
    usecols=["date", "sales"],
    parse_dates=["date"]
)

# Keep complete calendar months only
sales_data = sales_data[
    (sales_data["date"] >= "2013-01-01")
    & (sales_data["date"] < "2017-08-01")
].copy()

# Reconstruct monthly total sales
original_target = (
    sales_data
    .assign(
        month=lambda df:
            df["date"].dt.to_period("M").dt.to_timestamp()
    )
    .groupby("month", as_index=False)["sales"]
    .sum()
    .rename(columns={"sales": "original_total_sales"})
)

# Compare against the new modeling dataset
target_comparison = monthly_data[
    ["month", "total_sales"]
].merge(
    original_target,
    on="month",
    how="outer",
    validate="one_to_one",
    indicator=True
)

target_comparison["sales_difference"] = (
    target_comparison["total_sales"]
    - target_comparison["original_total_sales"]
)

display(target_comparison.head())

,month,total_sales,original_total_sales,_merge,sales_difference
0,2013-01-01,1.032762e+07,1.032762e+07,both,0.0
1,2013-02-01,9.658960e+06,9.658960e+06,both,0.0
2,2013-03-01,1.142850e+07,1.142850e+07,both,0.0
3,2013-04-01,1.099346e+07,1.099346e+07,both,0.0
4,2013-05-01,1.159770e+07,1.159770e+07,both,0.0


In [43]:
assert len(original_target) == 55
assert len(target_comparison) == 55

assert (target_comparison["_merge"] == "both").all()

assert target_comparison[
    ["total_sales", "original_total_sales"]
].notna().all().all()

assert np.allclose(
    target_comparison["total_sales"],
    target_comparison["original_total_sales"],
    rtol=0,
    atol=0.01
), "Monthly sales targets do not match."

print("Forecasting target reconciliation: PASSED")
print("Months compared:", len(target_comparison))

print(
    "Maximum absolute difference:",
    target_comparison["sales_difference"].abs().max()
)

print(
    "Total original sales:",
    round(original_target["original_total_sales"].sum(), 2)
)

print(
    "Total modeling sales:",
    round(monthly_data["total_sales"].sum(), 2)
)

Forecasting target reconciliation: PASSED
Months compared: 55
Maximum absolute difference: 0.0
Total original sales: 1061211629.26
Total modeling sales: 1061211629.26


## 5. Calendar-Aware Forecasting Model

### Model: Historical Seasonal-Index Forecast

This model combines:

1. The average monthly sales level from the most
   recent 12 completed training months.
2. Calendar-month seasonal indices estimated
   using complete years within the training data.

Forecast formula:

Forecast = Recent 12-Month Average Sales
           × Historical Seasonal Index

### Leakage Prevention

Seasonal indices and the sales baseline are
recalculated separately for every rolling origin,
using only data available at that origin.

No validation-period sales are used to construct
the forecasts.

In [44]:
def seasonal_index_forecast(train_data, validation_data):
    train = train_data.copy()
    validation = validation_data.copy()

    train["year"] = train["month"].dt.year
    train["calendar_month"] = train["month"].dt.month

    # Identify complete calendar years in training data
    complete_years = (
        train.groupby("year")["calendar_month"]
        .nunique()
    )

    complete_years = complete_years[
        complete_years == 12
    ].index

    seasonal_history = train[
        train["year"].isin(complete_years)
    ].copy()

    if seasonal_history.empty:
        raise ValueError(
            "No complete training years available."
        )

    # Each month's sales relative to its year's average
    seasonal_history["yearly_monthly_average"] = (
        seasonal_history.groupby("year")["total_sales"]
        .transform("mean")
    )

    seasonal_history["seasonal_ratio"] = (
        seasonal_history["total_sales"]
        / seasonal_history["yearly_monthly_average"]
    )

    seasonal_indices = (
        seasonal_history
        .groupby("calendar_month")["seasonal_ratio"]
        .mean()
    )

    # Average sales over the most recent 12 training months
    recent_sales_level = (
        train["total_sales"].tail(12).mean()
    )

    validation["calendar_month"] = (
        validation["month"].dt.month
    )

    validation["seasonal_index"] = (
        validation["calendar_month"].map(seasonal_indices)
    )

    validation["seasonal_index_forecast"] = (
        recent_sales_level
        * validation["seasonal_index"]
    )

    return validation

In [45]:
first_window = rolling_windows_df.iloc[0]

train_data = monthly_data[
    monthly_data["month"] <= first_window["train_end"]
].copy()

validation_data = monthly_data[
    (monthly_data["month"] >= first_window["validation_start"])
    & (monthly_data["month"] <= first_window["validation_end"])
].copy()

first_forecast = seasonal_index_forecast(
    train_data,
    validation_data
)

display(
    first_forecast[
        [
            "month",
            "total_sales",
            "seasonal_index",
            "seasonal_index_forecast"
        ]
    ]
)

,month,total_sales,seasonal_index,seasonal_index_forecast
36,2016-01-01,2.397780e+07,0.902695,1.812011e+07
37,2016-02-01,2.194741e+07,0.733227,1.471832e+07
38,2016-03-01,2.313178e+07,0.973471,1.954081e+07


In [46]:
assert len(first_forecast) == 3

assert first_forecast[
    "seasonal_index_forecast"
].notna().all()

assert (
    first_forecast["seasonal_index_forecast"] > 0
).all()

assert (
    first_forecast["month"]
    > first_window["train_end"]
).all()

print("First seasonal-index forecast validation: PASSED")
print("Training ends:", first_window["train_end"])
print("Validation begins:", first_window["validation_start"])
print("Validation ends:", first_window["validation_end"])

First seasonal-index forecast validation: PASSED
Training ends: 2015-12-01 00:00:00
Validation begins: 2016-01-01 00:00:00
Validation ends: 2016-03-01 00:00:00


## 6. Rolling-Origin Seasonal-Index Forecasting

### Objective

Evaluate the historical seasonal-index model
across all 17 rolling-origin windows.

For each window:

1. Select historical training observations.
2. Select the next three validation months.
3. Estimate seasonal indices from training data only.
4. Generate three monthly sales forecasts.
5. Store the forecast origin and predictions.

### Expected Results

- 17 rolling windows
- 3 forecasts per window
- 51 total forecast cases

No future actual sales are used to generate predictions.

In [47]:
seasonal_index_results = []

for _, window in rolling_windows_df.iterrows():

    train_data = monthly_data[
        monthly_data["month"] <= window["train_end"]
    ].copy()

    validation_data = monthly_data[
        (monthly_data["month"] >= window["validation_start"])
        & (monthly_data["month"] <= window["validation_end"])
    ].copy()

    forecast = seasonal_index_forecast(
        train_data,
        validation_data
    )

    forecast["window_id"] = window["window_id"]
    forecast["train_end"] = window["train_end"]

    seasonal_index_results.append(
        forecast[
            [
                "window_id",
                "train_end",
                "month",
                "total_sales",
                "seasonal_index",
                "seasonal_index_forecast"
            ]
        ]
    )

seasonal_index_forecasts_df = pd.concat(
    seasonal_index_results,
    ignore_index=True
)

display(seasonal_index_forecasts_df.head(10))

,window_id,train_end,month,total_sales,seasonal_index,seasonal_index_forecast
0,1,2015-12-01,2016-01-01,2.397780e+07,0.902695,1.812011e+07
1,1,2015-12-01,2016-02-01,2.194741e+07,0.733227,1.471832e+07
2,1,2015-12-01,2016-03-01,2.313178e+07,0.973471,1.954081e+07
3,2,2016-01-01,2016-02-01,2.194741e+07,0.733227,1.527319e+07
4,2,2016-01-01,2016-03-01,2.313178e+07,0.973471,2.027747e+07
5,2,2016-01-01,2016-04-01,2.596303e+07,0.807093,1.681182e+07
6,3,2016-02-01,2016-03-01,2.313178e+07,0.973471,2.094308e+07
7,3,2016-02-01,2016-04-01,2.596303e+07,0.807093,1.736367e+07
8,3,2016-02-01,2016-05-01,2.477943e+07,0.880293,1.893848e+07
9,4,2016-03-01,2016-04-01,2.596303e+07,0.807093,1.787033e+07


In [48]:
assert len(seasonal_index_forecasts_df) == 51

assert (
    seasonal_index_forecasts_df["window_id"].nunique()
    == 17
)

assert (
    seasonal_index_forecasts_df
    .groupby("window_id")
    .size()
    .eq(3)
    .all()
)

assert seasonal_index_forecasts_df[
    ["total_sales", "seasonal_index", "seasonal_index_forecast"]
].notna().all().all()

assert (
    seasonal_index_forecasts_df["seasonal_index_forecast"] > 0
).all()

assert (
    seasonal_index_forecasts_df["month"]
    > seasonal_index_forecasts_df["train_end"]
).all()

assert not seasonal_index_forecasts_df.duplicated(
    subset=["window_id", "month"]
).any()

print("Rolling seasonal-index forecast validation: PASSED")
print("Rolling windows:", seasonal_index_forecasts_df["window_id"].nunique())
print("Forecast cases:", len(seasonal_index_forecasts_df))
print("Missing forecasts:", seasonal_index_forecasts_df["seasonal_index_forecast"].isna().sum())
print("Duplicate window-month pairs:", seasonal_index_forecasts_df.duplicated(
    subset=["window_id", "month"]
).sum())

Rolling seasonal-index forecast validation: PASSED
Rolling windows: 17
Forecast cases: 51
Missing forecasts: 0
Duplicate window-month pairs: 0


## 7. Seasonal-Index Forecast Accuracy Evaluation

### Business Objective

Measure the accuracy of the seasonal-index model
using the same 51 rolling-origin forecast cases
as the existing forecasting baselines.

### Evaluation Metrics

- MAE: Mean Absolute Error
- RMSE: Root Mean Squared Error
- WAPE: Weighted Absolute Percentage Error
- Bias %: Signed Forecast Error / Actual Sales

### Error Convention

Forecast Error = Forecast - Actual Sales

Positive bias indicates overforecasting.
Negative bias indicates underforecasting.

In [49]:
seasonal_index_forecasts_df["forecast_error"] = (
    seasonal_index_forecasts_df["seasonal_index_forecast"]
    - seasonal_index_forecasts_df["total_sales"]
)

seasonal_index_forecasts_df["absolute_error"] = (
    seasonal_index_forecasts_df["forecast_error"].abs()
)

seasonal_index_forecasts_df["squared_error"] = (
    seasonal_index_forecasts_df["forecast_error"] ** 2
)

display(
    seasonal_index_forecasts_df[
        [
            "window_id",
            "month",
            "total_sales",
            "seasonal_index_forecast",
            "forecast_error",
            "absolute_error"
        ]
    ].head(10)
)

,window_id,month,total_sales,seasonal_index_forecast,forecast_error,absolute_error
0,1,2016-01-01,2.397780e+07,1.812011e+07,-5.857695e+06,5.857695e+06
1,1,2016-02-01,2.194741e+07,1.471832e+07,-7.229086e+06,7.229086e+06
2,1,2016-03-01,2.313178e+07,1.954081e+07,-3.590974e+06,3.590974e+06
3,2,2016-02-01,2.194741e+07,1.527319e+07,-6.674223e+06,6.674223e+06
4,2,2016-03-01,2.313178e+07,2.027747e+07,-2.854310e+06,2.854310e+06
5,2,2016-04-01,2.596303e+07,1.681182e+07,-9.151207e+06,9.151207e+06
6,3,2016-03-01,2.313178e+07,2.094308e+07,-2.188698e+06,2.188698e+06
7,3,2016-04-01,2.596303e+07,1.736367e+07,-8.599356e+06,8.599356e+06
8,3,2016-05-01,2.477943e+07,1.893848e+07,-5.840949e+06,5.840949e+06
9,4,2016-04-01,2.596303e+07,1.787033e+07,-8.092692e+06,8.092692e+06


In [50]:
actual = seasonal_index_forecasts_df["total_sales"]
error = seasonal_index_forecasts_df["forecast_error"]

seasonal_index_metrics = pd.DataFrame({
    "model": ["Seasonal Index"],

    "MAE": [error.abs().mean()],

    "RMSE": [np.sqrt((error ** 2).mean())],

    "WAPE_pct": [
        error.abs().sum() / actual.sum() * 100
    ],

    "Bias_pct": [
        error.sum() / actual.sum() * 100
    ]
})

display(seasonal_index_metrics.round(2))

,model,MAE,RMSE,WAPE_pct,Bias_pct
0,Seasonal Index,3591288.9,4136902.96,14.52,-8.09


In [51]:
seasonal_index_window_metrics = (
    seasonal_index_forecasts_df
    .groupby("window_id")
    .apply(
        lambda group: pd.Series({
            "train_end": group["train_end"].iloc[0],
            "validation_start": group["month"].min(),
            "validation_end": group["month"].max(),

            "MAE": group["absolute_error"].mean(),

            "RMSE": np.sqrt(
                group["squared_error"].mean()
            ),

            "WAPE_pct": (
                group["absolute_error"].sum()
                / group["total_sales"].sum()
                * 100
            ),

            "Bias_pct": (
                group["forecast_error"].sum()
                / group["total_sales"].sum()
                * 100
            )
        }),
        include_groups=False
    )
    .reset_index()
)

display(seasonal_index_window_metrics.head())

,window_id,train_end,validation_start,validation_end,MAE,RMSE,WAPE_pct,Bias_pct
0,1,2015-12-01,2016-01-01,2016-03-01,5.559252e+06,5.758107e+06,24.150711,-24.150711
1,2,2016-01-01,2016-02-01,2016-04-01,6.226580e+06,6.743810e+06,26.293859,-26.293859
2,3,2016-02-01,2016-03-01,2016-05-01,5.543001e+06,6.133406e+06,22.509881,-22.509881
3,4,2016-03-01,2016-04-01,2016-06-01,4.879403e+06,5.628458e+06,20.065624,-20.065624
4,5,2016-04-01,2016-05-01,2016-07-01,1.952661e+06,2.658654e+06,8.314938,-5.510075


In [52]:
assert len(seasonal_index_forecasts_df) == 51

assert len(seasonal_index_metrics) == 1

assert len(seasonal_index_window_metrics) == 17

assert seasonal_index_forecasts_df[
    ["forecast_error", "absolute_error", "squared_error"]
].notna().all().all()

assert np.isfinite(
    seasonal_index_metrics[
        ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
    ].to_numpy()
).all()

assert (
    seasonal_index_metrics["MAE"].iloc[0] >= 0
)

assert (
    seasonal_index_metrics["WAPE_pct"].iloc[0] >= 0
)

print("Seasonal-index accuracy evaluation: PASSED")
print("Forecast cases:", len(seasonal_index_forecasts_df))
print("Rolling windows:", len(seasonal_index_window_metrics))

print(
    "Overall WAPE:",
    round(seasonal_index_metrics["WAPE_pct"].iloc[0], 2),
    "%"
)

print(
    "Overall Bias:",
    round(seasonal_index_metrics["Bias_pct"].iloc[0], 2),
    "%"
)

Seasonal-index accuracy evaluation: PASSED
Forecast cases: 51
Rolling windows: 17
Overall WAPE: 14.52 %
Overall Bias: -8.09 %


## 8. Forecast Bias Diagnosis by Horizon

### Objective

Determine whether the seasonal-index model
systematically underforecasts sales at horizons
of one, two, and three months.

### Business Questions

- Which horizon has the highest WAPE?
- Is forecast bias consistently negative?
- Does forecast accuracy deteriorate at longer horizons?

### Method

Calculate WAPE and signed bias separately
for each forecast horizon.

In [53]:
# Identify forecast horizon within each rolling window
horizon_diagnostics = seasonal_index_forecasts_df.copy()

horizon_diagnostics["horizon"] = (
    horizon_diagnostics
    .groupby("window_id")
    .cumcount() + 1
)

horizon_metrics = (
    horizon_diagnostics
    .groupby("horizon")
    .agg(
        forecast_cases=("total_sales", "size"),
        actual_sales=("total_sales", "sum"),
        absolute_error=("absolute_error", "sum"),
        signed_error=("forecast_error", "sum")
    )
    .reset_index()
)

horizon_metrics["WAPE_pct"] = (
    horizon_metrics["absolute_error"]
    / horizon_metrics["actual_sales"]
    * 100
)

horizon_metrics["Bias_pct"] = (
    horizon_metrics["signed_error"]
    / horizon_metrics["actual_sales"]
    * 100
)

display(
    horizon_metrics[
        ["horizon", "forecast_cases", "WAPE_pct", "Bias_pct"]
    ].round(2)
)

,horizon,forecast_cases,WAPE_pct,Bias_pct
0,1,17,14.70,-7.99
1,2,17,14.74,-8.25
2,3,17,14.12,-8.03


In [54]:
assert horizon_metrics["horizon"].tolist() == [1, 2, 3]

assert (
    horizon_metrics["forecast_cases"] == 17
).all()

assert horizon_metrics["forecast_cases"].sum() == 51

assert np.isclose(
    horizon_metrics["absolute_error"].sum()
    / horizon_metrics["actual_sales"].sum() * 100,
    seasonal_index_metrics["WAPE_pct"].iloc[0]
)

assert np.isclose(
    horizon_metrics["signed_error"].sum()
    / horizon_metrics["actual_sales"].sum() * 100,
    seasonal_index_metrics["Bias_pct"].iloc[0]
)

print("Forecast horizon diagnostics: PASSED")

Forecast horizon diagnostics: PASSED


In [55]:
display(
    horizon_metrics[
        ["horizon", "forecast_cases", "WAPE_pct", "Bias_pct"]
    ].round(2)
)

,horizon,forecast_cases,WAPE_pct,Bias_pct
0,1,17,14.70,-7.99
1,2,17,14.74,-8.25
2,3,17,14.12,-8.03


## 9. Historical Sales Trend Diagnosis

### Objective

Investigate whether the seasonal-index model's
negative forecast bias could be associated with
changes in the underlying sales level.

### Method

For each rolling origin:

1. Calculate average monthly sales for the most
   recent 12 training months.
2. Calculate average monthly sales for the
   preceding 12 training months.
3. Calculate the percentage change between
   the two periods.

### Important

All calculations use training data only.
This analysis does not establish causation.

In [56]:
trend_diagnostics = []

for _, window in rolling_windows_df.iterrows():

    train = monthly_data[
        monthly_data["month"] <= window["train_end"]
    ].copy()

    recent_12 = train["total_sales"].iloc[-12:]
    previous_12 = train["total_sales"].iloc[-24:-12]

    recent_average = recent_12.mean()
    previous_average = previous_12.mean()

    growth_pct = (
        (recent_average / previous_average) - 1
    ) * 100

    trend_diagnostics.append({
        "window_id": window["window_id"],
        "train_end": window["train_end"],
        "previous_12m_average": previous_average,
        "recent_12m_average": recent_average,
        "growth_pct": growth_pct
    })

trend_diagnostics_df = pd.DataFrame(trend_diagnostics)

display(trend_diagnostics_df.round(2))

C:\Users\Dell\AppData\Local\Temp\ipykernel_4720\783678528.py:29: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(trend_diagnostics_df.round(2))


,window_id,train_end,previous_12m_average,recent_12m_average,growth_pct
0,1,2015-12-01,17456187.19,20073341.72,14.99
1,2,2016-01-01,17121627.30,20830081.96,21.66
2,3,2016-02-01,17263630.91,21513833.04,24.62
3,4,2016-03-01,16866382.86,22141597.48,31.28
4,5,2016-04-01,17040867.62,23058927.23,35.32
5,6,2016-05-01,17403416.21,23646349.23,35.87
6,7,2016-06-01,18094699.72,23695837.41,30.95
7,8,2016-07-01,18327010.39,23800258.53,29.86
8,9,2016-08-01,19083551.87,23757653.54,24.49
9,10,2016-09-01,19351757.40,23689034.01,22.41


In [57]:
assert len(trend_diagnostics_df) == 17

assert trend_diagnostics_df[
    [
        "previous_12m_average",
        "recent_12m_average",
        "growth_pct"
    ]
].notna().all().all()

assert (
    trend_diagnostics_df["previous_12m_average"] > 0
).all()

print("Historical sales trend diagnosis: PASSED")

print(
    "Windows with positive growth:",
    (trend_diagnostics_df["growth_pct"] > 0).sum()
)

print(
    "Average 12-month growth:",
    round(trend_diagnostics_df["growth_pct"].mean(), 2),
    "%"
)

print(
    "Minimum growth:",
    round(trend_diagnostics_df["growth_pct"].min(), 2),
    "%"
)

print(
    "Maximum growth:",
    round(trend_diagnostics_df["growth_pct"].max(), 2),
    "%"
)

Historical sales trend diagnosis: PASSED
Windows with positive growth: 17
Average 12-month growth: 22.36 %
Minimum growth: 6.9 %
Maximum growth: 35.87 %


## 10. Trend-Adjusted Seasonal-Index Forecast

### Objective

Test whether incorporating historical sales growth
improves the seasonal-index forecasting model.

### Formula

Forecast = Recent 12M Average
           × Historical Seasonal Index
           × Historical Growth Factor

Growth Factor = Recent 12M Average
                / Previous 12M Average

### Leakage Prevention

All calculations use data available at the
forecast origin.

### Limitation

Historical growth may not continue at the
same rate. Model accuracy must be evaluated
out of sample.

In [58]:
def trend_adjusted_seasonal_forecast(
    train_data,
    validation_data
):
    # Reuse our validated seasonal-index model
    forecast = seasonal_index_forecast(
        train_data,
        validation_data
    )

    # Calculate historical growth using training only
    recent_12m_avg = (
        train_data["total_sales"].iloc[-12:].mean()
    )

    previous_12m_avg = (
        train_data["total_sales"].iloc[-24:-12].mean()
    )

    if previous_12m_avg <= 0:
        raise ValueError(
            "Previous 12-month average must be positive."
        )

    growth_factor = recent_12m_avg / previous_12m_avg

    forecast["growth_factor"] = growth_factor

    forecast["trend_adjusted_forecast"] = (
        forecast["seasonal_index_forecast"]
        * growth_factor
    )

    return forecast

In [59]:
first_window = rolling_windows_df.iloc[0]

train_data = monthly_data[
    monthly_data["month"] <= first_window["train_end"]
].copy()

validation_data = monthly_data[
    (monthly_data["month"] >= first_window["validation_start"])
    & (monthly_data["month"] <= first_window["validation_end"])
].copy()

first_trend_forecast = trend_adjusted_seasonal_forecast(
    train_data,
    validation_data
)

display(
    first_trend_forecast[
        [
            "month",
            "total_sales",
            "seasonal_index_forecast",
            "growth_factor",
            "trend_adjusted_forecast"
        ]
    ].round(2)
)

C:\Users\Dell\AppData\Local\Temp\ipykernel_4720\545060186.py:26: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].round(2)


,month,total_sales,seasonal_index_forecast,growth_factor,trend_adjusted_forecast
36,2016-01-01,23977804.89,18120109.40,1.15,20836803.82
37,2016-02-01,21947408.80,14718322.98,1.15,16924997.62
38,2016-03-01,23131780.88,19540806.82,1.15,22470502.21


In [60]:
assert len(first_trend_forecast) == 3

assert first_trend_forecast[
    "trend_adjusted_forecast"
].notna().all()

assert (
    first_trend_forecast["trend_adjusted_forecast"] > 0
).all()

assert (
    first_trend_forecast["month"]
    > first_window["train_end"]
).all()

assert np.isclose(
    first_trend_forecast["growth_factor"].iloc[0],
    trend_diagnostics_df["recent_12m_average"].iloc[0]
    / trend_diagnostics_df["previous_12m_average"].iloc[0]
)

print("First trend-adjusted forecast validation: PASSED")

print(
    "Historical growth factor:",
    round(first_trend_forecast["growth_factor"].iloc[0], 4)
)

print(
    "Historical growth percentage:",
    round(
        (first_trend_forecast["growth_factor"].iloc[0] - 1)
        * 100,
        2
    ),
    "%"
)

First trend-adjusted forecast validation: PASSED
Historical growth factor: 1.1499
Historical growth percentage: 14.99 %


## 11. Rolling-Origin Trend-Adjusted Forecast Evaluation

### Objective

Evaluate the trend-adjusted seasonal-index model
using the same 17 rolling windows and 51 forecast
cases as the original seasonal-index model.

### Evaluation Metrics

- MAE
- RMSE
- WAPE
- Forecast Bias

### Business Question

Does incorporating historical sales growth
improve forecasting accuracy and reduce
systematic underforecasting?

In [61]:
trend_results = []

for _, window in rolling_windows_df.iterrows():

    train_data = monthly_data[
        monthly_data["month"] <= window["train_end"]
    ].copy()

    validation_data = monthly_data[
        (monthly_data["month"] >= window["validation_start"])
        & (monthly_data["month"] <= window["validation_end"])
    ].copy()

    forecast = trend_adjusted_seasonal_forecast(
        train_data,
        validation_data
    )

    forecast["window_id"] = window["window_id"]
    forecast["train_end"] = window["train_end"]

    trend_results.append(
        forecast[
            [
                "window_id",
                "train_end",
                "month",
                "total_sales",
                "seasonal_index_forecast",
                "growth_factor",
                "trend_adjusted_forecast"
            ]
        ]
    )

trend_forecasts_df = pd.concat(
    trend_results,
    ignore_index=True
)

display(trend_forecasts_df.head(10))

,window_id,train_end,month,total_sales,seasonal_index_forecast,growth_factor,trend_adjusted_forecast
0,1,2015-12-01,2016-01-01,2.397780e+07,1.812011e+07,1.149927,2.083680e+07
1,1,2015-12-01,2016-02-01,2.194741e+07,1.471832e+07,1.149927,1.692500e+07
2,1,2015-12-01,2016-03-01,2.313178e+07,1.954081e+07,1.149927,2.247050e+07
3,2,2016-01-01,2016-02-01,2.194741e+07,1.527319e+07,1.216595,1.858128e+07
4,2,2016-01-01,2016-03-01,2.313178e+07,2.027747e+07,1.216595,2.466947e+07
5,2,2016-01-01,2016-04-01,2.596303e+07,1.681182e+07,1.216595,2.045317e+07
6,3,2016-02-01,2016-03-01,2.313178e+07,2.094308e+07,1.246194,2.609914e+07
7,3,2016-02-01,2016-04-01,2.596303e+07,1.736367e+07,1.246194,2.163850e+07
8,3,2016-02-01,2016-05-01,2.477943e+07,1.893848e+07,1.246194,2.360102e+07
9,4,2016-03-01,2016-04-01,2.596303e+07,1.787033e+07,1.312765,2.345955e+07


In [62]:
trend_forecasts_df["forecast_error"] = (
    trend_forecasts_df["trend_adjusted_forecast"]
    - trend_forecasts_df["total_sales"]
)

trend_forecasts_df["absolute_error"] = (
    trend_forecasts_df["forecast_error"].abs()
)

trend_forecasts_df["squared_error"] = (
    trend_forecasts_df["forecast_error"] ** 2
)

actual = trend_forecasts_df["total_sales"]
error = trend_forecasts_df["forecast_error"]

trend_metrics = pd.DataFrame({
    "model": ["Trend-Adjusted Seasonal Index"],

    "MAE": [error.abs().mean()],

    "RMSE": [
        np.sqrt((error ** 2).mean())
    ],

    "WAPE_pct": [
        error.abs().sum() / actual.sum() * 100
    ],

    "Bias_pct": [
        error.sum() / actual.sum() * 100
    ]
})

display(trend_metrics.round(2))

,model,MAE,RMSE,WAPE_pct,Bias_pct
0,Trend-Adjusted Seasonal Index,4786033.22,6151133.27,19.35,12.58


In [63]:
assert len(trend_forecasts_df) == 51

assert trend_forecasts_df["window_id"].nunique() == 17

assert (
    trend_forecasts_df.groupby("window_id")
    .size()
    .eq(3)
    .all()
)

assert not trend_forecasts_df.duplicated(
    subset=["window_id", "month"]
).any()

assert (
    trend_forecasts_df["month"]
    > trend_forecasts_df["train_end"]
).all()

assert trend_forecasts_df[
    [
        "total_sales",
        "growth_factor",
        "trend_adjusted_forecast",
        "forecast_error"
    ]
].notna().all().all()

assert np.isfinite(
    trend_metrics[
        ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
    ].to_numpy()
).all()

# Confirm both models use exactly the same evaluation cases
comparison_keys = ["window_id", "month"]

original_cases = (
    seasonal_index_forecasts_df[comparison_keys]
    .sort_values(comparison_keys)
    .reset_index(drop=True)
)

trend_cases = (
    trend_forecasts_df[comparison_keys]
    .sort_values(comparison_keys)
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    original_cases,
    trend_cases
)

print("Trend-adjusted rolling forecast validation: PASSED")
print("Rolling windows:", trend_forecasts_df["window_id"].nunique())
print("Forecast cases:", len(trend_forecasts_df))

print(
    "Trend-adjusted WAPE:",
    round(trend_metrics["WAPE_pct"].iloc[0], 2),
    "%"
)

print(
    "Trend-adjusted Bias:",
    round(trend_metrics["Bias_pct"].iloc[0], 2),
    "%"
)

Trend-adjusted rolling forecast validation: PASSED
Rolling windows: 17
Forecast cases: 51
Trend-adjusted WAPE: 19.35 %
Trend-adjusted Bias: 12.58 %


In [64]:
seasonal_model_comparison = pd.concat(
    [
        seasonal_index_metrics,
        trend_metrics
    ],
    ignore_index=True
)

display(
    seasonal_model_comparison.round(2)
)

,model,MAE,RMSE,WAPE_pct,Bias_pct
0,Seasonal Index,3591288.90,4136902.96,14.52,-8.09
1,Trend-Adjusted Seasonal Index,4786033.22,6151133.27,19.35,12.58


## 12. Partial Trend Adjustment — Exploratory Analysis

### Objective

Investigate whether applying only part of historical
sales growth reduces forecasting error.

### Candidate Adjustment Strengths

- 0%: No trend adjustment
- 25%: Partial adjustment
- 50%: Moderate adjustment
- 75%: Strong adjustment
- 100%: Full adjustment

### Important

These results are exploratory because the same
rolling-validation cases are used to compare
adjustment strengths.

Selecting the best strength and reporting its
accuracy on these same cases would introduce
model-selection bias.

A separate outer evaluation is required before
claiming a generalizable improvement.

In [65]:
adjustment_results = []

for alpha in [0, 0.25, 0.50, 0.75, 1.0]:

    adjusted_forecast = (
        trend_forecasts_df["seasonal_index_forecast"]
        * (
            1 + alpha
            * (trend_forecasts_df["growth_factor"] - 1)
        )
    )

    actual = trend_forecasts_df["total_sales"]
    error = adjusted_forecast - actual

    adjustment_results.append({
        "trend_weight": alpha,
        "MAE": error.abs().mean(),
        "RMSE": np.sqrt((error ** 2).mean()),
        "WAPE_pct": error.abs().sum() / actual.sum() * 100,
        "Bias_pct": error.sum() / actual.sum() * 100
    })

partial_trend_comparison = pd.DataFrame(
    adjustment_results
)

display(partial_trend_comparison.round(2))

,trend_weight,MAE,RMSE,WAPE_pct,Bias_pct
0,0.00,3591288.90,4136902.96,14.52,-8.09
1,0.25,3734540.07,4069451.03,15.10,-2.92
2,0.50,3957620.96,4453051.31,16.00,2.24
3,0.75,4272810.96,5188609.54,17.27,7.41
4,1.00,4786033.22,6151133.27,19.35,12.58


In [66]:
assert len(partial_trend_comparison) == 5

assert np.isclose(
    partial_trend_comparison.iloc[0]["WAPE_pct"],
    seasonal_index_metrics.iloc[0]["WAPE_pct"]
)

assert np.isclose(
    partial_trend_comparison.iloc[-1]["WAPE_pct"],
    trend_metrics.iloc[0]["WAPE_pct"]
)

assert np.isfinite(
    partial_trend_comparison[
        ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
    ].to_numpy()
).all()

print("Partial trend exploration: PASSED")

Partial trend exploration: PASSED


## 13. Calendar-Month Forecast Error Analysis

### Objective

Identify which calendar months contribute most
to the seasonal-index model's forecasting errors.

### Metrics

- Number of forecast cases
- Actual sales
- Total absolute error
- WAPE
- Forecast bias

### Important

Some calendar months appear in multiple rolling
windows. Results represent forecast cases rather
than independent monthly observations.

In [67]:
calendar_error_data = seasonal_index_forecasts_df.copy()

calendar_error_data["calendar_month"] = (
    calendar_error_data["month"].dt.month
)

calendar_error_metrics = (
    calendar_error_data
    .groupby("calendar_month")
    .agg(
        forecast_cases=("total_sales", "size"),
        actual_sales=("total_sales", "sum"),
        absolute_error=("absolute_error", "sum"),
        signed_error=("forecast_error", "sum")
    )
    .reset_index()
)

calendar_error_metrics["WAPE_pct"] = (
    calendar_error_metrics["absolute_error"]
    / calendar_error_metrics["actual_sales"]
    * 100
)

calendar_error_metrics["Bias_pct"] = (
    calendar_error_metrics["signed_error"]
    / calendar_error_metrics["actual_sales"]
    * 100
)

calendar_error_metrics["month_name"] = (
    pd.to_datetime(
        calendar_error_metrics["calendar_month"],
        format="%m"
    ).dt.month_name()
)

display(
    calendar_error_metrics[
        [
            "month_name",
            "forecast_cases",
            "WAPE_pct",
            "Bias_pct"
        ]
    ].round(2)
)

,month_name,forecast_cases,WAPE_pct,Bias_pct
0,January,4,19.07,-19.07
1,February,5,25.15,-25.15
2,March,6,12.19,-12.19
3,April,6,25.33,-25.33
4,May,6,18.53,-18.53
5,June,5,5.77,-5.48
6,July,4,5.81,2.94
7,August,3,3.60,3.60
8,September,3,16.99,16.99
9,October,3,10.83,10.83


In [68]:
assert len(calendar_error_metrics) == 12

assert calendar_error_metrics["forecast_cases"].sum() == 51

assert np.isclose(
    calendar_error_metrics["absolute_error"].sum()
    / calendar_error_metrics["actual_sales"].sum() * 100,
    seasonal_index_metrics["WAPE_pct"].iloc[0]
)

assert np.isclose(
    calendar_error_metrics["signed_error"].sum()
    / calendar_error_metrics["actual_sales"].sum() * 100,
    seasonal_index_metrics["Bias_pct"].iloc[0]
)

print("Calendar-month forecast error analysis: PASSED")

Calendar-month forecast error analysis: PASSED


## 14. Calendar-Month Forecast Error Contribution

### Objective

Identify which calendar months contribute the
largest share of total absolute forecasting error.

### Business Questions

- Which months create the greatest total forecast error?
- Do the highest-WAPE months also contribute the
  most absolute error?
- Which months should receive priority for
  forecasting improvements?

### Important

The analysis counts overlapping rolling-origin
forecast cases, not independent calendar months.

In [69]:
error_contribution = calendar_error_metrics.copy()

total_absolute_error = (
    error_contribution["absolute_error"].sum()
)

error_contribution["error_contribution_pct"] = (
    error_contribution["absolute_error"]
    / total_absolute_error
    * 100
)

error_contribution = (
    error_contribution
    .sort_values(
        "absolute_error",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    error_contribution[
        [
            "month_name",
            "forecast_cases",
            "absolute_error",
            "WAPE_pct",
            "error_contribution_pct"
        ]
    ].round(2)
)

,month_name,forecast_cases,absolute_error,WAPE_pct,error_contribution_pct
0,April,6,39410063.56,25.33,21.52
1,May,6,28737809.55,18.53,15.69
2,February,5,28580353.46,25.15,15.60
3,January,4,19633202.24,19.07,10.72
4,March,6,18224790.53,12.19,9.95
5,September,3,11426392.88,16.99,6.24
6,December,3,8498156.00,9.56,4.64
7,October,3,7807410.39,10.83,4.26
8,June,5,6803484.59,5.77,3.71
9,November,3,5953604.62,8.05,3.25


In [70]:
assert len(error_contribution) == 12

assert np.isclose(
    error_contribution["error_contribution_pct"].sum(),
    100
)

assert np.isclose(
    error_contribution["absolute_error"].sum(),
    seasonal_index_forecasts_df["absolute_error"].sum()
)

assert (
    error_contribution["absolute_error"]
    .is_monotonic_decreasing
)

print("Calendar-month error contribution: PASSED")

print(
    "Top three months' combined error contribution:",
    round(
        error_contribution["error_contribution_pct"]
        .head(3)
        .sum(),
        2
    ),
    "%"
)

Calendar-month error contribution: PASSED
Top three months' combined error contribution: 52.81 %


## 15. Business Findings and Forecasting Recommendations

### 1. Forecast Evaluation Design

- Monthly sales observations: 55
- Rolling-origin windows: 17
- Forecast horizon: 3 months
- Total forecast cases: 51
- Validation period: January 2016 to July 2017
- Monthly sales target reconciliation: PASSED
- No future actual sales used to generate forecasts

### 2. Seasonal-Index Model Performance

- MAE: 3,591,288.90
- RMSE: 4,136,902.96
- WAPE: 14.52%
- Forecast bias: -8.09%

The model systematically underforecast sales.

### 3. Historical Sales Growth

- Positive-growth rolling windows: 17 of 17
- Average historical 12-month growth: 22.36%
- Minimum growth: 6.90%
- Maximum growth: 35.87%

Historical growth was positive but varied substantially
between forecasting origins.

### 4. Trend Adjustment Results

| Trend Weight | WAPE | Bias |
|---|---:|---:|
| 0% | 14.52% | -8.09% |
| 25% | 15.10% | -2.92% |
| 50% | 16.00% | +2.24% |
| 75% | 17.27% | +7.41% |
| 100% | 19.35% | +12.58% |

Applying historical growth increased WAPE.

The 25% adjustment reduced absolute bias and slightly
improved RMSE, but did not improve WAPE.

### 5. Forecast Horizon Analysis

| Horizon | WAPE | Bias |
|---|---:|---:|
| 1 month | 14.70% | -7.99% |
| 2 months | 14.74% | -8.25% |
| 3 months | 14.12% | -8.03% |

Underforecasting occurred consistently across all
three forecast horizons.

### 6. Calendar-Month Error Concentration

Highest absolute-error contributors:

1. April: 21.52%
2. May: 15.69%
3. February: 15.60%

Combined contribution: 52.81%.

January through May contributed approximately 73.48%
of total absolute forecast error.

### 7. Retail Business Recommendations

- Retain the unadjusted seasonal-index model as the
  stronger WAPE performer among the tested trend weights.
- Prioritize investigation of January–May forecasting errors.
- Do not assume historical sales growth will continue.
- Evaluate any future calendar or promotion features
  using leakage-safe rolling-origin validation.
- Compare new models against existing naive, seasonal
  naive, blended, and ETS benchmarks before deployment.

### 8. Analytical Limitations

- Forecast cases overlap across rolling windows.
- Calendar-month groups contain only 3–6 forecast cases.
- Trend weights were explored using the same validation
  cases, so their results are exploratory.
- Correlation and historical growth patterns do not
  establish causation.
- The effect of promotion features on forecast accuracy
  has not yet been evaluated.
- A final model recommendation requires matched-case
  comparison with existing forecasting benchmarks.

### Final Conclusion

Adding historical growth to the seasonal-index model
did not improve WAPE.

The unadjusted seasonal-index model achieved 14.52%
WAPE, while the full trend-adjusted model achieved
19.35% WAPE.

Further forecasting improvements should focus on
validated calendar features, changing demand patterns,
and operational drivers rather than assuming that
more complex models are automatically better.

## 16. Export Forecasting Evaluation Reports

### Objective

Export validated forecasting results for:

- Model performance comparison
- Rolling-origin forecasts
- Forecast horizon diagnostics
- Calendar-month error analysis
- Historical sales growth
- Partial trend adjustment analysis

### Output Location

reports/forecasting_feature_evaluation/

### Data Governance

All reports must preserve consistent column names,
forecast definitions, evaluation windows, and
reproducible analytical results.

In [71]:
from pathlib import Path

report_dir = (
    PROJECT_ROOT
    / "reports"
    / "forecasting_feature_evaluation"
)

report_dir.mkdir(
    parents=True,
    exist_ok=True
)

reports_to_export = {
    "seasonal_model_comparison.csv":
        seasonal_model_comparison,

    "seasonal_index_rolling_forecasts.csv":
        seasonal_index_forecasts_df,

    "trend_adjusted_rolling_forecasts.csv":
        trend_forecasts_df,

    "forecast_horizon_diagnostics.csv":
        horizon_metrics,

    "calendar_month_forecast_errors.csv":
        calendar_error_metrics,

    "calendar_month_error_contribution.csv":
        error_contribution,

    "historical_sales_trend_diagnostics.csv":
        trend_diagnostics_df,

    "partial_trend_adjustment_comparison.csv":
        partial_trend_comparison
}

for filename, dataframe in reports_to_export.items():

    output_path = report_dir / filename

    dataframe.to_csv(
        output_path,
        index=False,
        float_format="%.10f"
    )

    print(
        f"Exported: {filename} "
        f"({len(dataframe)} rows)"
    )

print("\nTotal reports:", len(reports_to_export))

Exported: seasonal_model_comparison.csv (2 rows)
Exported: seasonal_index_rolling_forecasts.csv (51 rows)
Exported: trend_adjusted_rolling_forecasts.csv (51 rows)
Exported: forecast_horizon_diagnostics.csv (3 rows)
Exported: calendar_month_forecast_errors.csv (12 rows)
Exported: calendar_month_error_contribution.csv (12 rows)
Exported: historical_sales_trend_diagnostics.csv (17 rows)
Exported: partial_trend_adjustment_comparison.csv (5 rows)

Total reports: 8


In [72]:
expected_rows = {
    "seasonal_model_comparison.csv": 2,
    "seasonal_index_rolling_forecasts.csv": 51,
    "trend_adjusted_rolling_forecasts.csv": 51,
    "forecast_horizon_diagnostics.csv": 3,
    "calendar_month_forecast_errors.csv": 12,
    "calendar_month_error_contribution.csv": 12,
    "historical_sales_trend_diagnostics.csv": 17,
    "partial_trend_adjustment_comparison.csv": 5
}

for filename, expected_count in expected_rows.items():

    file_path = report_dir / filename

    assert file_path.exists(), (
        f"Missing report: {filename}"
    )

    exported = pd.read_csv(file_path)

    assert len(exported) == expected_count, (
        f"Incorrect row count: {filename}"
    )

    assert exported.columns.tolist() == (
        reports_to_export[filename].columns.tolist()
    ), f"Column mismatch: {filename}"

    print(
        f"PASS: {filename} "
        f"({len(exported)} rows)"
    )

print("\nForecasting report export validation: PASSED")
print("Reports validated:", len(expected_rows))
print("Total report rows:", sum(expected_rows.values()))

PASS: seasonal_model_comparison.csv (2 rows)
PASS: seasonal_index_rolling_forecasts.csv (51 rows)
PASS: trend_adjusted_rolling_forecasts.csv (51 rows)
PASS: forecast_horizon_diagnostics.csv (3 rows)
PASS: calendar_month_forecast_errors.csv (12 rows)
PASS: calendar_month_error_contribution.csv (12 rows)
PASS: historical_sales_trend_diagnostics.csv (17 rows)
PASS: partial_trend_adjustment_comparison.csv (5 rows)

Forecasting report export validation: PASSED
Reports validated: 8
Total report rows: 153


## 17. Existing Forecast Benchmark Inspection

### Objective

Inspect the original forecasting notebook to identify:

- Existing benchmark model names
- Forecast output variables
- Forecast result columns
- Rolling-origin evaluation structure

### Governance Rule

All models must be compared using identical
forecast origins and target months.

In [73]:
import ast

benchmark_notebook = notebook_data

for cell_number, cell in enumerate(
    benchmark_notebook["cells"], start=1
):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if any(
        keyword in source.lower()
        for keyword in [
            "forecast_results",
            "forecast_df",
            "predictions",
            "rolling_forecast",
            "model_comparison"
        ]
    ):
        print(f"\n{'=' * 60}")
        print(f"CELL {cell_number}")
        print(f"{'=' * 60}")
        print(source[:1200])


CELL 60
rolling_forecasts = []

for window_id, window in rolling_windows_df.iterrows():

    train_end = window["train_end"]
    validation_start = window["validation_start"]
    validation_end = window["validation_end"]

    # Training data
    train_data = monthly_ts[
        monthly_ts["month"] <= train_end
    ].copy()

    # Validation data
    validation_data = monthly_ts[
        (monthly_ts["month"] >= validation_start)
        & (monthly_ts["month"] <= validation_end)
    ].copy()

    # -------------------------
    # Naive forecast
    # -------------------------
    naive_value = train_data["total_sales"].iloc[-1]

    validation_data["naive_forecast"] = naive_value

    # -------------------------
    # Seasonal Naive forecast
    # -------------------------
    seasonal_reference_dates = (
        validation_data["month"]
        - pd.DateOffset(years=1)
    )

    seasonal_reference = (
        monthly_ts[
            monthly_ts["month"].isin(seasonal_reference_dates)
 

## 18. Existing Benchmark Forecast Report Discovery

### Objective

Identify reusable benchmark forecast CSV files.

### Validation Requirement

Forecasts must match on window ID, training end date,
validation month, and actual sales before comparing metrics.

In [74]:
from pathlib import Path

report_root = PROJECT_ROOT / "reports"

forecast_files = sorted(
    path
    for path in report_root.rglob("*.csv")
    if any(
        keyword in path.name.lower()
        for keyword in [
            "forecast",
            "rolling",
            "benchmark",
            "model_comparison"
        ]
    )
)

print("Forecast-related CSV files:", len(forecast_files))

for path in forecast_files:
    print(path.relative_to(PROJECT_ROOT))

Forecast-related CSV files: 5
reports\forecasting_feature_evaluation\calendar_month_forecast_errors.csv
reports\forecasting_feature_evaluation\forecast_horizon_diagnostics.csv
reports\forecasting_feature_evaluation\seasonal_index_rolling_forecasts.csv
reports\forecasting_feature_evaluation\seasonal_model_comparison.csv
reports\forecasting_feature_evaluation\trend_adjusted_rolling_forecasts.csv


## 19. Original Benchmark Output Inspection

### Objective

Determine whether the original forecasting notebook
contains saved benchmark model results.

### Models

- Naive
- Seasonal Naive
- 50/50 Blended
- ETS Additive
- ETS Damped Trend

### Governance

Saved notebook outputs must be inspected and validated
before they are reused for model comparison.

In [77]:
benchmark_output_cells = [
    60, 109, 115, 124, 127, 133, 136, 137
]

for cell_number in benchmark_output_cells:

    cell = benchmark_notebook["cells"][cell_number - 1]

    outputs = cell.get("outputs", [])

    print(f"\nCELL {cell_number}")
    print("Saved outputs:", len(outputs))

    for output in outputs:

        data = output.get("data", {})

        if "text/plain" in data:

            output_text = "".join(
                data["text/plain"]
            )

            print(output_text[:500])

        elif "text" in output:

            print(
                "".join(output["text"])[:500]
            )


CELL 60
Saved outputs: 1
   window_id  train_end      month   total_sales  naive_forecast  \
0          1 2015-12-01 2016-01-01  2.397780e+07    2.724398e+07   
1          1 2015-12-01 2016-02-01  2.194741e+07    2.724398e+07   
2          1 2015-12-01 2016-03-01  2.313178e+07    2.724398e+07   
3          2 2016-01-01 2016-02-01  2.194741e+07    2.397780e+07   
4          2 2016-01-01 2016-03-01  2.313178e+07    2.397780e+07   
5          2 2016-01-01 2016-04-01  2.596303e+07    2.397780e+07   
6          3 2016-02-01 

CELL 109
Saved outputs: 1
   window_id      month   total_sales  naive_forecast  \
0          1 2016-01-01  2.397780e+07    2.724398e+07   
1          1 2016-02-01  2.194741e+07    2.724398e+07   
2          1 2016-03-01  2.313178e+07    2.724398e+07   
3          2 2016-02-01  2.194741e+07    2.397780e+07   
4          2 2016-03-01  2.313178e+07    2.397780e+07   
5          2 2016-04-01  2.596303e+07    2.397780e+07   
6          3 2016-03-01  2.313178e+07    2.1947

## 20. Matched-Case Forecast Benchmark Comparison

### Business Objective

Compare five established forecasting models against
the seasonal-index and trend-adjusted seasonal-index
models using identical rolling-origin forecast cases.

### Models

1. Naive
2. Seasonal Naive
3. 50/50 Blended
4. ETS Additive
5. ETS Damped Trend
6. Seasonal Index
7. Trend-Adjusted Seasonal Index

### Data Governance

Before comparing model performance, validate:

- Identical rolling-origin window IDs
- Identical training end dates
- Identical forecast months
- Identical actual sales
- No missing or duplicate forecast cases

### Primary Metric

WAPE (Weighted Absolute Percentage Error)

### Supporting Metrics

MAE, RMSE and Forecast Bias

In [78]:
benchmark_path = (
    PROJECT_ROOT
    / "reports"
    / "forecasting_feature_evaluation"
    / "original_benchmark_rolling_forecasts.csv"
)

benchmark_df = pd.read_csv(
    benchmark_path,
    parse_dates=["train_end", "month"]
)

print("Benchmark forecast cases:", len(benchmark_df))

display(benchmark_df.head())

Benchmark forecast cases: 51


,window_id,train_end,month,total_sales,naive_forecast,seasonal_naive_forecast,blended_forecast,ets_forecast,damped_ets_forecast
0,1,2015-12-01,2016-01-01,2.397780e+07,2.724398e+07,1.489692e+07,2.107045e+07,2.346665e+07,2.262673e+07
1,1,2015-12-01,2016-02-01,2.194741e+07,2.724398e+07,1.374240e+07,2.049319e+07,1.930117e+07,1.910118e+07
2,1,2015-12-01,2016-03-01,2.313178e+07,2.724398e+07,1.559861e+07,2.142129e+07,2.421934e+07,2.330344e+07
3,2,2016-01-01,2016-02-01,2.194741e+07,2.397780e+07,1.374240e+07,1.886010e+07,1.936486e+07,1.959551e+07
4,2,2016-01-01,2016-03-01,2.313178e+07,2.397780e+07,1.559861e+07,1.978821e+07,2.428877e+07,2.446527e+07


In [79]:
new_forecasts_df = (
    trend_forecasts_df[
        [
            "window_id",
            "train_end",
            "month",
            "total_sales",
            "seasonal_index_forecast",
            "trend_adjusted_forecast"
        ]
    ]
    .rename(
        columns={
            "total_sales": "new_actual_sales"
        }
    )
)

matched_forecasts_df = benchmark_df.merge(
    new_forecasts_df,
    on=["window_id", "train_end", "month"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

print(
    "Match status:"
)

print(
    matched_forecasts_df["_merge"].value_counts()
)

display(matched_forecasts_df.head())

Match status:
_merge
both          51
left_only      0
right_only     0
Name: count, dtype: int64


,window_id,train_end,month,total_sales,naive_forecast,seasonal_naive_forecast,blended_forecast,ets_forecast,damped_ets_forecast,new_actual_sales,seasonal_index_forecast,trend_adjusted_forecast,_merge
0,1,2015-12-01,2016-01-01,2.397780e+07,2.724398e+07,1.489692e+07,2.107045e+07,2.346665e+07,2.262673e+07,2.397780e+07,1.812011e+07,2.083680e+07,both
1,1,2015-12-01,2016-02-01,2.194741e+07,2.724398e+07,1.374240e+07,2.049319e+07,1.930117e+07,1.910118e+07,2.194741e+07,1.471832e+07,1.692500e+07,both
2,1,2015-12-01,2016-03-01,2.313178e+07,2.724398e+07,1.559861e+07,2.142129e+07,2.421934e+07,2.330344e+07,2.313178e+07,1.954081e+07,2.247050e+07,both
3,2,2016-01-01,2016-02-01,2.194741e+07,2.397780e+07,1.374240e+07,1.886010e+07,1.936486e+07,1.959551e+07,2.194741e+07,1.527319e+07,1.858128e+07,both
4,2,2016-01-01,2016-03-01,2.313178e+07,2.397780e+07,1.559861e+07,1.978821e+07,2.428877e+07,2.446527e+07,2.313178e+07,2.027747e+07,2.466947e+07,both


In [80]:
assert len(matched_forecasts_df) == 51

assert matched_forecasts_df["_merge"].eq("both").all(), (
    "Some forecast cases do not match."
)

assert matched_forecasts_df["window_id"].nunique() == 17

assert not matched_forecasts_df.duplicated(
    ["window_id", "train_end", "month"]
).any()

assert np.allclose(
    matched_forecasts_df["total_sales"],
    matched_forecasts_df["new_actual_sales"],
    rtol=0,
    atol=0.01
), "Actual sales differ between forecasting notebooks."

assert (
    matched_forecasts_df["train_end"]
    < matched_forecasts_df["month"]
).all()

print("Matched-case forecast validation: PASSED")
print("Matched forecast cases:", len(matched_forecasts_df))
print("Rolling-origin windows:", matched_forecasts_df["window_id"].nunique())

Matched-case forecast validation: PASSED
Matched forecast cases: 51
Rolling-origin windows: 17


## 21. Seven-Model Forecast Performance Comparison

### Objective

Evaluate seven forecasting models using identical
rolling-origin forecast cases.

### Evaluation Metrics

- MAE: Average absolute forecast error
- RMSE: Root mean squared forecast error
- WAPE: Total absolute error / total actual sales
- Bias: Total signed error / total actual sales

### Forecast Error Convention

Forecast Error = Forecast - Actual

Negative bias indicates underforecasting.

Positive bias indicates overforecasting.

### Model Selection

Use WAPE as the primary selection metric,
supported by RMSE and forecast bias.

In [81]:
model_columns = {
    "Naive": "naive_forecast",
    "Seasonal Naive": "seasonal_naive_forecast",
    "50/50 Blended": "blended_forecast",
    "ETS Additive": "ets_forecast",
    "ETS Damped Trend": "damped_ets_forecast",
    "Seasonal Index": "seasonal_index_forecast",
    "Trend-Adjusted Seasonal Index": "trend_adjusted_forecast"
}

actual = matched_forecasts_df["total_sales"]

comparison_rows = []

for model_name, forecast_column in model_columns.items():

    forecast = matched_forecasts_df[forecast_column]

    error = forecast - actual

    comparison_rows.append({
        "model": model_name,
        "MAE": error.abs().mean(),
        "RMSE": np.sqrt((error ** 2).mean()),
        "WAPE_pct": (
            error.abs().sum() / actual.abs().sum()
        ) * 100,
        "Bias_pct": (
            error.sum() / actual.abs().sum()
        ) * 100
    })

seven_model_comparison_df = (
    pd.DataFrame(comparison_rows)
    .sort_values("WAPE_pct")
    .reset_index(drop=True)
)

display(seven_model_comparison_df.round(2))

,model,MAE,RMSE,WAPE_pct,Bias_pct
0,50/50 Blended,2067404.61,2737131.72,8.36,-6.45
1,ETS Additive,2098242.04,2542024.93,8.48,0.96
2,ETS Damped Trend,2118859.66,2549228.14,8.56,0.30
3,Naive,2178014.34,2735640.76,8.80,-0.59
4,Seasonal Naive,3210528.38,4529398.49,12.98,-12.31
5,Seasonal Index,3591288.90,4136902.96,14.52,-8.09
6,Trend-Adjusted Seasonal Index,4786033.22,6151133.27,19.35,12.58


In [82]:
assert len(seven_model_comparison_df) == 7

assert seven_model_comparison_df["model"].nunique() == 7

assert seven_model_comparison_df[
    ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
].notna().all().all()

assert np.isfinite(
    seven_model_comparison_df[
        ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
    ].to_numpy()
).all()

assert (
    seven_model_comparison_df["WAPE_pct"] >= 0
).all()

assert seven_model_comparison_df[
    "WAPE_pct"
].is_monotonic_increasing

print("Seven-model comparison validation: PASSED")

print(
    "Best model by WAPE:",
    seven_model_comparison_df.iloc[0]["model"]
)

print(
    "Best WAPE:",
    round(
        seven_model_comparison_df.iloc[0]["WAPE_pct"],
        2
    ),
    "%"
)

Seven-model comparison validation: PASSED
Best model by WAPE: 50/50 Blended
Best WAPE: 8.36 %


## 22. Historical Benchmark Reconciliation

### Objective

Verify that recalculated forecast metrics match
the original forecasting notebook.

### Acceptance Criteria

- All five original benchmark models are present.
- WAPE and bias match historical values within
  0.01 percentage points.
- MAE and RMSE match historical values within
  0.02 sales units.

### Governance

Historical results are comparison references,
not replacements for recalculated metrics.

In [83]:
historical_metrics = pd.DataFrame([
    ["Naive", 2178014.34, 2735640.76, 8.80, -0.59],
    ["Seasonal Naive", 3210528.38, 4529398.49, 12.98, -12.31],
    ["50/50 Blended", 2067404.61, 2737131.72, 8.36, -6.45],
    ["ETS Additive", 2098242.04, 2542024.93, 8.48, 0.96],
    ["ETS Damped Trend", 2118859.66, 2549228.14, 8.56, 0.30]
], columns=[
    "model", "historical_MAE", "historical_RMSE",
    "historical_WAPE_pct", "historical_Bias_pct"
])

reconciliation_df = historical_metrics.merge(
    seven_model_comparison_df,
    on="model",
    how="left",
    validate="one_to_one"
)

for metric in ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]:
    reconciliation_df[f"{metric}_difference"] = (
        reconciliation_df[metric]
        - reconciliation_df[f"historical_{metric}"]
    )

display(reconciliation_df[
    [
        "model",
        "MAE_difference",
        "RMSE_difference",
        "WAPE_pct_difference",
        "Bias_pct_difference"
    ]
].round(4))

,model,MAE_difference,RMSE_difference,WAPE_pct_difference,Bias_pct_difference
0,Naive,0.0004,0.0041,0.0039,-0.0027
1,Seasonal Naive,0.0011,-0.0031,-0.0026,-0.0005
2,50/50 Blended,-0.0005,-0.0025,-0.0032,-0.0016
3,ETS Additive,0.0009,0.0039,0.0014,0.0012
4,ETS Damped Trend,0.0016,-0.0006,0.0047,0.0048


In [84]:
assert len(reconciliation_df) == 5

assert reconciliation_df[
    ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
].notna().all().all()

assert (
    reconciliation_df["MAE_difference"].abs() <= 0.02
).all()

assert (
    reconciliation_df["RMSE_difference"].abs() <= 0.02
).all()

assert (
    reconciliation_df["WAPE_pct_difference"].abs() <= 0.01
).all()

assert (
    reconciliation_df["Bias_pct_difference"].abs() <= 0.01
).all()

print("Historical benchmark reconciliation: PASSED")
print("Models reconciled:", len(reconciliation_df))

Historical benchmark reconciliation: PASSED
Models reconciled: 5


## 23. Final Seven-Model Comparison Export

### Objective

Save the validated seven-model forecasting results
for reporting, Power BI integration, and GitHub documentation.

### Deliverables

1. Seven-model performance comparison
2. Matched rolling-origin forecast predictions

### Governance

Preserve full metric precision in the exported CSV files.

Validate the files after exporting.

In [85]:
from pathlib import Path

report_dir = (
    PROJECT_ROOT
    / "reports"
    / "forecasting_feature_evaluation"
)

report_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Output 1: Model performance
comparison_path = (
    report_dir
    / "seven_model_forecast_comparison.csv"
)

seven_model_comparison_df.to_csv(
    comparison_path,
    index=False,
    float_format="%.10f"
)

# Output 2: Individual matched forecasts
matched_path = (
    report_dir
    / "seven_model_matched_forecasts.csv"
)

matched_export_df = matched_forecasts_df.drop(
    columns=["_merge", "new_actual_sales"]
)

matched_export_df.to_csv(
    matched_path,
    index=False,
    float_format="%.10f"
)

print("Final forecast reports exported successfully.")
print("Model comparison rows:", len(seven_model_comparison_df))
print("Matched forecast rows:", len(matched_export_df))
print("Files:")
print(comparison_path.name)
print(matched_path.name)

Final forecast reports exported successfully.
Model comparison rows: 7
Matched forecast rows: 51
Files:
seven_model_forecast_comparison.csv
seven_model_matched_forecasts.csv


In [86]:
comparison_check = pd.read_csv(comparison_path)

matched_check = pd.read_csv(
    matched_path,
    parse_dates=["train_end", "month"]
)

assert len(comparison_check) == 7
assert len(matched_check) == 51

assert comparison_check["model"].nunique() == 7

assert matched_check["window_id"].nunique() == 17

assert not matched_check.duplicated(
    ["window_id", "train_end", "month"]
).any()

assert matched_check[
    list(model_columns.values()) + ["total_sales"]
].notna().all().all()

assert np.allclose(
    comparison_check[
        ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
    ].to_numpy(),
    seven_model_comparison_df[
        ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
    ].to_numpy(),
    rtol=0,
    atol=1e-6
)

assert np.allclose(
    matched_check[
        list(model_columns.values()) + ["total_sales"]
    ].to_numpy(),
    matched_export_df[
        list(model_columns.values()) + ["total_sales"]
    ].to_numpy(),
    rtol=0,
    atol=1e-6
)

print("Final seven-model CSV export validation: PASSED")
print("Models:", len(comparison_check))
print("Forecast cases:", len(matched_check))

Final seven-model CSV export validation: PASSED
Models: 7
Forecast cases: 51



## 24. Final Forecasting Model Evaluation — Business Conclusions

### 24.1 Business Objective

Evaluate forecasting methods that can support retail sales planning, budgeting, inventory decisions, and operational resource allocation.

Seven models were compared using 17 expanding rolling-origin windows and 51 three-month-ahead forecast cases.

The evaluation covers January 2016 through July 2017, with overlapping validation periods.

### 24.2 Model Performance

| Model | WAPE | Bias |
|---|---:|---:|
| 50/50 Blended | 8.36% | -6.45% |
| ETS Additive | 8.48% | +0.96% |
| ETS Damped Trend | 8.56% | +0.30% |
| Naive | 8.80% | -0.59% |
| Seasonal Naive | 12.98% | -12.31% |
| Seasonal Index | 14.52% | -8.09% |
| Trend-Adjusted Seasonal Index | 19.35% | +12.58% |

### 24.3 Key Findings

**Finding 1 — The blended model achieved the lowest WAPE.**

The 50/50 blended model achieved 8.36% WAPE, outperforming the other six models on the primary evaluation metric.

However, its -6.45% bias indicates systematic underforecasting across the evaluated cases.

**Finding 2 — ETS models provided more balanced forecasts.**

ETS Additive achieved 8.48% WAPE, +0.96% bias, and the lowest RMSE of approximately 2.54 million.

ETS Damped Trend achieved 8.56% WAPE and +0.30% bias.

These models may be preferable when avoiding systematic underforecasting is important.

**Finding 3 — Additional trend adjustment did not improve accuracy.**

The seasonal-index model achieved 14.52% WAPE.

Applying the full historical growth adjustment increased WAPE to 19.35% and changed bias from -8.09% to +12.58%.

This suggests that directly extrapolating historical growth was too aggressive for these validation periods.

**Finding 4 — Greater model complexity did not guarantee better performance.**

The simple 50/50 blended model achieved lower WAPE than the ETS and seasonal-index alternatives.

Model complexity should therefore be justified by measurable improvements in forecasting performance.

### 24.4 Business Recommendation

**Primary candidate based on WAPE:** 50/50 Blended.

**Alternative candidate for more balanced forecast errors:** ETS Additive.

Before operational deployment, compare these candidates on a new, untouched evaluation period and assess the business cost of underforecasting and overforecasting.

### 24.5 Limitations

- Forecasting is performed on aggregate monthly sales, not individual products or stores.
- The 51 forecast cases contain overlapping calendar months and are not 51 independent observations.
- Results are historical backtesting performance, not guaranteed future accuracy.
- The 50/50 blending rule was evaluated and selected using the same backtesting cases.
- Promotions, holidays, prices, and other external drivers are not directly incorporated as future-known predictors in these seven models.
- Forecast accuracy does not directly measure inventory availability, stockouts, or replenishment performance.

### 24.6 Final Conclusion

The matched-case evaluation identified the 50/50 blended model as the strongest historical performer by WAPE, while ETS Additive offered a competitive accuracy–bias balance.

The results support further evaluation of both models before selecting a production forecasting approach.
